# Generalization tests: pair/cold-isolated auxiliary deltaG

Based on `../05_generalization_tests.ipynb`. The auxiliary deltaG model excludes
all structures of every held-out (Diene_Index, Ene_Index) pair. For cold diene,
cold ene, and cold both tests, its training data also excludes the corresponding
held-out reactants, including mixed reactions omitted from the both-cold test.
These exclusions apply to both active-only and broad deltaG training, regardless
of whether an Interaction label is present.

Grouped folds keep every pair together even when its rows are nonconsecutive.
Barrier-ordered folds remain a diagnostic, with pair isolation for the auxiliary
model. Training-row deltaG predictions remain in-sample, as in the original;
this notebook does not add inner out-of-fold stacking.

Results are displayed in memory only. CatBoost file writing is disabled; no
CSV, figure, or model export is enabled.


In [1]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [2]:
import numpy as np
import pandas as pd
import _modeling_common as modeling_common
from _modeling_common import (PHYSORG_FEATURE_COUNT, TARGET_COLUMNS,
    evaluate_targets, load_modeling_data, metrics, new_catboost)

# Disable implicit catboost_info / snapshot files for BOTH model stages.
# This changes the configuration in memory only, not _modeling_common.py.
modeling_common.CATBOOST_PARAMS = {
    **modeling_common.CATBOOST_PARAMS, "allow_writing_files": False,
}

data_Ga, X_Ga, data_G, X_G = load_modeling_data(DATA_DIR, DESCRIPTOR_MAP)
Y_DD = data_Ga["Diene_Distort"].to_numpy()
Y_ED = data_Ga["Ene_Distort"].to_numpy()
Y_IN = data_Ga["Interaction"].to_numpy()
Y_Ga = data_Ga["deltaGa(Solvent)"].to_numpy()
Y_G = data_Ga["deltaG"].to_numpy()
Y_G_data_G = data_G["deltaG"].to_numpy()
print(f"Aligned active-energy rows: {len(data_Ga):,}; broad rows: {len(data_G):,}")


100%|██████████| 50161/50161 [00:07<00:00, 6942.02it/s]

Aligned active-energy rows: 8,273; broad rows: 50,161


In [3]:
VARIANTS = ["PhysOrg", "PhysOrg + SVO",
            "PhysOrg + SVO + predicted reaction energy (active only)",
            "PhysOrg + SVO + predicted reaction energy (broad)"]

def reactant_split(frame, seed, holdout):
    """Mirror OOD_train_test in 3_model_train.ipynb."""
    np.random.seed(seed)
    diene_indexs = np.sort(np.unique(frame["Diene_Index"].to_numpy()))
    ene_indexs = np.sort(np.unique(frame["Ene_Index"].to_numpy()))
    np.random.shuffle(diene_indexs)
    np.random.shuffle(ene_indexs)
    removed_dienes = diene_indexs[:len(diene_indexs) // 10]
    removed_enes = ene_indexs[:len(ene_indexs) // 10]

    if holdout == "both":
        train_ids = [row_id for row_id, row in frame.iterrows()
                     if row["Diene_Index"] not in removed_dienes
                     and row["Ene_Index"] not in removed_enes]
        test_ids = [row_id for row_id, row in frame.iterrows()
                    if row["Diene_Index"] in removed_dienes
                    and row["Ene_Index"] in removed_enes]
    elif holdout == "diene":
        train_ids = [row_id for row_id, row in frame.iterrows()
                     if row["Diene_Index"] not in removed_dienes]
        test_ids = [row_id for row_id, row in frame.iterrows()
                    if row["Diene_Index"] in removed_dienes]
    elif holdout == "ene":
        train_ids = [row_id for row_id, row in frame.iterrows()
                     if row["Ene_Index"] not in removed_enes]
        test_ids = [row_id for row_id, row in frame.iterrows()
                    if row["Ene_Index"] in removed_enes]
    else:
        raise ValueError(holdout)
    return train_ids, test_ids
def pair_keys(frame):
    # Structure_id is deliberately omitted: isolate the entire reactant pair.
    return pd.MultiIndex.from_frame(frame[["Diene_Index", "Ene_Index"]])


def auxiliary_training_ids(data_g, data_ga, train_ids, test_ids,
                           source="broad", cold=None):
    """Apply outer pair/cold exclusions before fitting the deltaG model."""
    if cold not in (None, "diene", "ene", "both"):
        raise ValueError(f"Unknown cold split: {cold}")
    train_ids = np.asarray(train_ids, dtype=int)
    test_ids = np.asarray(test_ids, dtype=int)
    if np.intersect1d(train_ids, test_ids).size:
        raise ValueError("Outer training and test rows overlap")
    if source == "broad":
        frame = data_g
        keep = np.ones(len(frame), dtype=bool)
    elif source == "active":
        frame = data_ga
        keep = np.zeros(len(frame), dtype=bool)
        keep[train_ids] = True
    else:
        raise ValueError(f"Unknown first-stage source: {source}")

    held_out_pairs = pair_keys(data_ga.iloc[test_ids]).unique()
    # No Interaction.isna() exception: unlabeled structures must also be excluded.
    keep &= ~pair_keys(frame).isin(held_out_pairs)
    cold_columns = {None: (), "diene": ("Diene_Index",),
                    "ene": ("Ene_Index",),
                    "both": ("Diene_Index", "Ene_Index")}[cold]
    held_out_reactants = {}
    for column in cold_columns:
        # Derive from ALL outer rows missing from training, not just test rows.
        # This includes mixed reactions excluded from the both-cold evaluation.
        held_out = set(data_ga[column]) - set(data_ga.iloc[train_ids][column])
        held_out_reactants[column] = held_out
        keep &= ~frame[column].isin(held_out).to_numpy()

    ids = np.flatnonzero(keep)
    if len(ids) < 2:
        raise ValueError("Too few isolated auxiliary deltaG training rows")
    assert not pair_keys(frame.iloc[ids]).isin(held_out_pairs).any()
    for column, held_out in held_out_reactants.items():
        assert not frame.iloc[ids][column].isin(held_out).any()
    return ids


def fit_delta_g_feature(data_g, x_g, data_ga, x_ga, train_ids,
                        test_ids, source="broad", seed=0, cold=None):
    g_ids = auxiliary_training_ids(data_g, data_ga, train_ids, test_ids,
                                   source=source, cold=cold)
    frame, x = (data_g, x_g) if source == "broad" else (data_ga, x_ga)
    model = new_catboost(seed)
    model.fit(x[g_ids], frame["deltaG"].to_numpy()[g_ids])
    return model.predict(x_ga)


def evaluate_splits(splits, split_type, cold=None):
    results = []
    for split_id, (train_ids, test_ids) in enumerate(splits):
        if len(train_ids) < 2 or len(test_ids) < 2:
            raise ValueError(f"Split {split_id} has too few rows")
        for variant in VARIANTS:
            if variant == "PhysOrg":
                features = X_Ga[:, :PHYSORG_FEATURE_COUNT]
            elif variant == "PhysOrg + SVO":
                features = X_Ga
            else:
                source = "active" if variant.endswith("(active only)") else "broad"
                pred_g = fit_delta_g_feature(data_G, X_G, data_Ga, X_Ga,
                                             train_ids, test_ids, source=source, cold=cold)
                features = np.column_stack([X_Ga, pred_g])
            rows, _ = evaluate_targets(data_Ga, features, train_ids, test_ids)
            results.extend({"split_type": split_type, "split": split_id,
                            "n_train": len(train_ids), "n_test": len(test_ids),
                            "variant": variant, **row} for row in rows)
    return pd.DataFrame(results)

In [4]:
ood_results = []
for holdout in ["diene", "ene", "both"]:
    splits = [reactant_split(data_Ga, seed, holdout) for seed in [0, 3, 6]]
    ood_results.append(evaluate_splits(splits, f"{holdout} reactant OOD", cold=holdout))
ood_results = pd.concat(ood_results, ignore_index=True)
display(ood_results.groupby(["split_type", "variant", "target"])[["r2", "mae"]].mean())

r2  \
split_type         variant                                            target                                     
both reactant OOD  PhysOrg                                            Aqueous activation free energy  0.587607   
                                                                      Diene distortion                0.619068   
                                                                      Ene distortion                  0.486310   
                                                                      Interaction                     0.509077   
                   PhysOrg + SVO                                      Aqueous activation free energy  0.663823   
                                                                      Diene distortion                0.674330   
                                                                      Ene distortion                  0.548385   
                                                                      Interaction                     0.561620   
                   PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  0.658501   
                                                                      Diene distortion                0.695517   
                                                                      Ene distortion                  0.571470   
                                                                      Interaction                     0.575975   
                   PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  0.659429   
                                                                      Diene distortion                0.696099   
                                                                      Ene distortion                  0.549220   
                                                                      Interaction                     0.594853   
diene reactant OOD PhysOrg                                            Aqueous activation free energy  0.608663   
                                                                      Diene distortion                0.612269   
                                                                      Ene distortion                  0.575228   
                                                                      Interaction                     0.639907   
                   PhysOrg + SVO                                      Aqueous activation free energy  0.663932   
                                                                      Diene distortion                0.651625   
                                                                      Ene distortion                  0.626465   
                                                                      Interaction                     0.658265   
                   PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  0.676765   
                                                                      Diene distortion                0.680881   
                                                                      Ene distortion                  0.629725   
                                                                      Interaction                     0.660071   
                   PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  0.657864   
                                                                      Diene distortion                0.669650   
                                                                      Ene distortion                  0.596800   
                                                                      Interaction                     0.660969   
ene reactant OOD   PhysOrg                                            Aqueous activation free energy  0.783999   
                                                                      Diene distortion                0.849187   
    

## Grouped reaction folds


In [5]:
from sklearn.model_selection import KFold


def grouped_pair_folds(frame, n_splits=5, seed=0):
    # Group by pair globally, rather than assuming equal pairs are adjacent.
    group_ids, pairs = pd.factorize(pair_keys(frame), sort=True)
    folds = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_groups, test_groups in folds.split(np.arange(len(pairs))):
        train_ids = np.flatnonzero(np.isin(group_ids, train_groups))
        test_ids = np.flatnonzero(np.isin(group_ids, test_groups))
        assert not pair_keys(frame.iloc[train_ids]).isin(
            pair_keys(frame.iloc[test_ids])).any()
        yield train_ids, test_ids


grouped_folds = list(grouped_pair_folds(data_Ga, 5, 0))
grouped_results = evaluate_splits(grouped_folds, "grouped reaction fold")
display(grouped_results.groupby(["variant", "target"])[["r2", "mae"]].mean())


r2  \
variant                                            target                                     
PhysOrg                                            Aqueous activation free energy  0.828825   
                                                   Diene distortion                0.878494   
                                                   Ene distortion                  0.673962   
                                                   Interaction                     0.671395   
PhysOrg + SVO                                      Aqueous activation free energy  0.846996   
                                                   Diene distortion                0.887775   
                                                   Ene distortion                  0.721078   
                                                   Interaction                     0.703744   
PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  0.845541   
                                                   Diene distortion                0.888642   
                                                   Ene distortion                  0.721816   
                                                   Interaction                     0.704319   
PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  0.863315   
                                                   Diene distortion                0.897193   
                                                   Ene distortion                  0.735898   
                                                   Interaction                     0.705862   

                                                                                        mae  
variant                                            target                                    
PhysOrg                                            Aqueous activation free energy  2.369195  
                                                   Diene distortion                1.541879  
                                                   Ene distortion                  1.613914  
                                                   Interaction                     2.034156  
PhysOrg + SVO                                      Aqueous activation free energy  2.224262  
                                                   Diene distortion                1.473909  
                                                   Ene distortion                  1.480833  
                                                   Interaction                     1.909508  
PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  2.220881  
                                                   Diene distortion                1.457526  
                                                   Ene distortion                  1.472569  
                                                   Interaction                     1.907273  
PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  2.112485  
                                                   Diene distortion                1.407849  
                                                   Ene distortion                  1.436835  
                                                   Interaction                     1.899467